In [12]:
import sqlite3
import numpy as np
import pandas as pd
import nflreadpy as nfl
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss

# ---- 1. Load your existing feature table ----
conn = sqlite3.connect("../data/nfl.db")
df_full = pd.read_sql_query("SELECT * FROM games_with_features", conn)
conn.close()
df_full['home_win'] = (df_full['home_score'] > df_full['away_score']).astype(int)

# ---- 2. Compute the new defense-process rating ----
TEAM_CODE_MAP = {'SD': 'LAC', 'STL': 'LA', 'OAK': 'LV'}

def standardize_team_codes(df):
    df = df.copy()
    df['home_team_std'] = df['home_team'].replace(TEAM_CODE_MAP)
    df['away_team_std'] = df['away_team'].replace(TEAM_CODE_MAP)
    return df

RAW_SCORE_WEIGHTS = {
    'def_sacks': 1.0,
    'def_qb_hits': 0.5,
    'def_tackles_for_loss': 0.5,
    'def_interceptions': 2.0,
    'def_pass_defended': 0.5,
}

def add_defense_process_rating(df_full, team_defense_stats, k=0.15, revert_fraction=1/3):
    df_full = standardize_team_codes(df_full)

    stats = team_defense_stats[
        ['game_id', 'season', 'week', 'team'] + list(RAW_SCORE_WEIGHTS.keys())
    ].copy()
    stats['team_std'] = stats['team'].replace(TEAM_CODE_MAP)
    stats['raw_score'] = sum(stats[col] * w for col, w in RAW_SCORE_WEIGHTS.items())

    league_avg = stats['raw_score'].mean()
    all_teams = set(stats['team_std'])
    ratings = {team: league_avg for team in all_teams}
    current_season = None

    stats = stats.sort_values(['season', 'week']).reset_index(drop=True)
    pre_values = []
    for _, row in stats.iterrows():
        if current_season is not None and row['season'] != current_season:
            for team in ratings:
                ratings[team] = league_avg + (ratings[team] - league_avg) * (1 - revert_fraction)
        current_season = row['season']

        team = row['team_std']
        pre_values.append(ratings[team])
        ratings[team] = ratings[team] + k * (row['raw_score'] - ratings[team])

    stats['pass_rush_coverage_rating_pre'] = pre_values

    rating_merge = stats[['game_id', 'team_std', 'pass_rush_coverage_rating_pre']]
    home_r = rating_merge.rename(columns={'team_std': 'home_team_std', 'pass_rush_coverage_rating_pre': 'home_pass_rush_coverage_rating_pre'})
    away_r = rating_merge.rename(columns={'team_std': 'away_team_std', 'pass_rush_coverage_rating_pre': 'away_pass_rush_coverage_rating_pre'})

    df_full = df_full.drop(columns=['home_pass_rush_coverage_rating_pre', 'away_pass_rush_coverage_rating_pre'], errors='ignore')
    df_full = df_full.merge(home_r, on=['game_id', 'home_team_std'], how='left')
    df_full = df_full.merge(away_r, on=['game_id', 'away_team_std'], how='left')

    df_full['defense_process_advantage'] = (
        df_full['home_pass_rush_coverage_rating_pre'] - df_full['away_pass_rush_coverage_rating_pre']
    )
    return df_full

seasons = sorted(df_full['season'].dropna().unique().astype(int).tolist())
team_stats = nfl.load_team_stats(seasons=seasons).to_pandas()

df_full = add_defense_process_rating(df_full, team_stats)
print("Nulls in new feature:", df_full['defense_process_advantage'].isna().sum(), "of", len(df_full))

# ---- 3. Held-out comparison: current 61 features vs. +defense_process_advantage ----
BASE_FEATURE_COLS = [
    'home_recent_form', 'away_recent_form',
    'home_recent_point_diff', 'away_recent_point_diff',
    'home_qb_recent_yards', 'away_qb_recent_yards',
    'home_qb_recent_tds', 'away_qb_recent_tds',
    'home_qb_recent_ints', 'away_qb_recent_ints',
    'home_qb_recent_epa', 'away_qb_recent_epa',
    'home_rb_recent_rush_yards', 'away_rb_recent_rush_yards',
    'home_rb_recent_rush_epa', 'away_rb_recent_rush_epa',
    'home_rb_recent_rec_yards', 'away_rb_recent_rec_yards',
    'home_wrte_recent_rec_yards', 'away_wrte_recent_rec_yards',
    'home_wrte_recent_rec_epa', 'away_wrte_recent_rec_epa',
    'home_wrte_recent_targets', 'away_wrte_recent_targets',
    'home_qb_injury_flag', 'away_qb_injury_flag',
    'home_rb_injury_flag', 'away_rb_injury_flag',
    'home_wrte_injury_flag', 'away_wrte_injury_flag',
    'home_epa_allowed_recent', 'away_epa_allowed_recent',
    'home_yards_allowed_recent', 'away_yards_allowed_recent',
    'home_takeaways_recent', 'away_takeaways_recent',
    'home_coach_h2h_wins', 'h2h_games_played',
    'home_off_rating_pre', 'home_def_rating_pre',
    'away_off_rating_pre', 'away_def_rating_pre',
    'rest_advantage',
    'home_sack_rate_recent', 'away_sack_rate_recent',
    'home_pressure_pct_recent', 'away_pressure_pct_recent',
    'home_wr_height_advantage', 'away_wr_height_advantage',
    'home_wr_weight_advantage', 'away_wr_weight_advantage',
    'home_opp_cb_completion_allowed', 'away_opp_cb_completion_allowed',
    'home_opp_cb_rating_allowed', 'away_opp_cb_rating_allowed',
    'home_star_rb_injured', 'away_star_rb_injured',
    'home_star_wr_injured', 'away_star_wr_injured',
    'div_game', 'spread_line',
]

def evaluate(feature_cols, df_full, train_ids, test_ids, label):
    train = df_full[df_full['game_id'].isin(train_ids)]
    test = df_full[df_full['game_id'].isin(test_ids)]
    X_train, y_train = train[feature_cols], train['home_win']
    X_test, y_test = test[feature_cols], test['home_win']

    imputer = SimpleImputer(strategy='mean')
    X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=feature_cols)
    X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=feature_cols)

    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    model.fit(X_train_imp, y_train)

    probs = model.predict_proba(X_test_imp)[:, 1]
    preds = (probs > 0.5).astype(int)

    print(f"--- {label} ({len(feature_cols)} features) ---")
    print(f"Accuracy:  {accuracy_score(y_test, preds):.4f}")
    print(f"Log loss:  {log_loss(y_test, probs):.4f}")
    print(f"Brier:     {brier_score_loss(y_test, probs):.4f}")
    print()

game_ids = np.asarray(df_full['game_id'].unique(), dtype=object)
train_ids, test_ids = train_test_split(game_ids, test_size=0.2, random_state=42)

evaluate(BASE_FEATURE_COLS, df_full, train_ids, test_ids, "WITHOUT defense_process_advantage")
evaluate(BASE_FEATURE_COLS + ['defense_process_advantage'], df_full, train_ids, test_ids, "WITH defense_process_advantage")

Nulls in new feature: 0 of 3028
--- WITHOUT defense_process_advantage (61 features) ---
Accuracy:  0.6469
Log loss:  0.6335
Brier:     0.2218

--- WITH defense_process_advantage (62 features) ---
Accuracy:  0.6485
Log loss:  0.6335
Brier:     0.2218



In [13]:
from sklearn.model_selection import KFold

def evaluate_cv(feature_cols, df_full, game_ids, label, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    accs, losses, briers = [], [], []

    for train_idx, test_idx in kf.split(game_ids):
        train_ids, test_ids = game_ids[train_idx], game_ids[test_idx]
        train = df_full[df_full['game_id'].isin(train_ids)]
        test = df_full[df_full['game_id'].isin(test_ids)]
        X_train, y_train = train[feature_cols], train['home_win']
        X_test, y_test = test[feature_cols], test['home_win']

        imputer = SimpleImputer(strategy='mean')
        X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=feature_cols)
        X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=feature_cols)

        model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
        model.fit(X_train_imp, y_train)

        probs = model.predict_proba(X_test_imp)[:, 1]
        preds = (probs > 0.5).astype(int)

        accs.append(accuracy_score(y_test, preds))
        losses.append(log_loss(y_test, probs))
        briers.append(brier_score_loss(y_test, probs))

    print(f"--- {label} ({len(feature_cols)} features, {n_splits}-fold CV) ---")
    print(f"Accuracy:  {np.mean(accs):.4f} (+/- {np.std(accs):.4f})")
    print(f"Log loss:  {np.mean(losses):.4f} (+/- {np.std(losses):.4f})")
    print(f"Brier:     {np.mean(briers):.4f} (+/- {np.std(briers):.4f})")
    print()

evaluate_cv(BASE_FEATURE_COLS, df_full, game_ids, "WITHOUT defense_process_advantage")
evaluate_cv(BASE_FEATURE_COLS + ['defense_process_advantage'], df_full, game_ids, "WITH defense_process_advantage")

--- WITHOUT defense_process_advantage (61 features, 5-fold CV) ---
Accuracy:  0.6460 (+/- 0.0124)
Log loss:  0.6279 (+/- 0.0107)
Brier:     0.2189 (+/- 0.0050)

--- WITH defense_process_advantage (62 features, 5-fold CV) ---
Accuracy:  0.6473 (+/- 0.0118)
Log loss:  0.6283 (+/- 0.0108)
Brier:     0.2191 (+/- 0.0050)

